# Four-way factor benchmark — CD8 T cells (GenePT text-embedding-3-large prior)

Identical to `four_way_benchmark_cd8.ipynb` except the SemanticSCVI prior comes from **GenePT** (`text-embedding-3-large` embeddings of NCBI+UniProt gene summaries, 3072-d, Zenodo 10833191) instead of Geneformer V2. LDVAE+ / scHPF / cNMF reuse the cached models under `.model_cache_cd8/` so the only thing that changes vs the geneformer run is `semantic_geom`.

All architecture / training knobs are explicit in **Cell 2 (parameters)** — kept bit-identical to the geneformer notebook.

In [ ]:
# ============================================================
# Parameters — edit here. All training/architecture knobs in one place.
# ============================================================
from pathlib import Path

# Anchor every path to notebooks/. Works whether jupyter was launched from
# notebooks/, the project root, or a parent dir of the project.
def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(
        f"Could not locate benchmark_helpers.py from cwd={Path.cwd()}. "
        "Launch jupyter under the scvi-tools-neural-nmf tree, or set NB_DIR manually."
    )

NB_DIR = _find_nb_dir()
print(f"NB_DIR = {NB_DIR}")

ADATA_PATH = NB_DIR / "zheng_cd8_clean.h5ad"
# GenePT artifacts (raw pickle + per-adata aligned tensor cache).
SEMANTIC_CACHE = NB_DIR / "zheng_cd8_clean_genept_3large.pt"
GENEPT_PICKLE_PATH = NB_DIR / "GenePT_gene_protein_embedding_model_3_text.pickle"
PATHWAY_INDEX = NB_DIR / "pathway_index.pkl"
LIB1_GMT = NB_DIR / "lib1_immune.gmt"
LIB2_GMT = NB_DIR / "lib2_cd8.gmt"
OUT_DIR = NB_DIR / "benchmark_results" / "cd8_four_way_genept"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preprocessing ----
HVG_TOP_N = None       # None = use all genes; int = take top-N HVGs (sc.pp.highly_variable_genes)
HVG_FLAVOR = "seurat_v3"  # works on raw counts; switch to "seurat" / "cell_ranger" if you log-normalized

# ---- Cache / retrain control ----
# Same parent cache dir as the geneformer notebook so LDVAE+/scHPF/NMF cache-hit
# the existing trained artifacts. Only SemanticSCVI uses a separate subdir.
MODEL_CACHE_DIR = NB_DIR / ".model_cache_cd8"
SEMANTIC_SCVI_SUBDIR = "semantic_scvi_genept"
FORCE_TRAIN_SEMANTIC = True  # ← retrain SemanticSCVI even if cached
FORCE_TRAIN_LDVAE = False
FORCE_TRAIN_SCHPF = False
FORCE_TRAIN_NMF = False

# ---- Shared model size ----
N_LATENT = 10

# ---- SemanticSCVI training ----
# use_batch_norm flows into BOTH encoder and decoder on the upstream LDVAE.
# Set to False to drop batch norm from the linear decoder (and the encoder).
SEMANTIC_MAX_EPOCHS = 200
SEMANTIC_WARMUP_EPOCHS = 30
SEMANTIC_N_EPOCHS_KL_WARMUP = 50  # linear KL anneal 0→1 over N epochs; finishes shortly after semantic loss turns on
SEMANTIC_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=1500.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- LDVAE+ training ----
LDVAE_MAX_EPOCHS = 250
LDVAE_KWARGS = dict(
    n_hidden=128,
    n_latent=N_LATENT,
    n_layers=1,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- scHPF / cNMF ----
N_FACTORS = N_LATENT
NMF_INIT = "nndsvd"
NMF_MAX_ITER = 500
NMF_RANDOM_STATE = 42

# ---- Benchmark / report ----
MODEL_NAMES = ["semantic_geom", "ldvae_nn", "schpf_k10", "nmf_k10"]
PER_PROJECTION_N_TOP = 30
CLUSTER_N_TOP = 500     # pool of top-loaded genes fed into UMAP+Leiden and hierarchical clustering
CLUSTER_TOP_PER_CLUSTER = 30  # int = after Leiden, keep only top-N genes per cluster (by max_loading); None = no gating
GENE_MAPPING = ("feature_id", "feature_name")  # Ensembl → symbol via adata.var columns
ENABLE_LLM_GRADING = True  # set False to skip Sonnet/Haiku scoring

In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import scanpy as sc
import torch
from sklearn.decomposition import NMF
from scipy.sparse import coo_matrix, issparse
import numpy as np
import schpf

from benchmark_helpers import (
    NMFWrapper,
    _ScviAdapter,
    build_report,
    train_or_load_nonneg_ldvae,
    train_or_load_pickle,
    train_or_load_semantic_scvi,
)
from benchmarking import SemanticBenchmark
from train_schpf import train_nmf_model, train_schpf_model

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NB_DIR = {NB_DIR}")

In [ ]:
# ============================================================
# GenePT semantic-map loader (inline; mirrors get_or_build_geneformer_map).
# Downloads the Zenodo v2 bundle on first call, extracts the
# text-embedding-3-large pickle, aligns rows to adata.var via HGNC symbols.
# ============================================================
import pickle
import urllib.request
import zipfile

GENEPT_ZENODO_URL = (
    "https://zenodo.org/records/10833191/files/GenePT_emebdding_v2.zip"
)
GENEPT_PICKLE_NAME = "GenePT_gene_protein_embedding_model_3_text.pickle"


def get_or_build_genept_map(
    adata,
    cache_path,
    pickle_path,
    *,
    symbol_key="feature_name",
    fill_missing="zero",
    zenodo_url=GENEPT_ZENODO_URL,
    pickle_basename=GENEPT_PICKLE_NAME,
):
    """Build/load a GenePT (text-embedding-3-large) semantic map for an AnnData.

    Rows align 1:1 to adata.var; lookup uses uppercased HGNC symbols
    (GenePT pickle uses UPPER-CASE HGNC keys). Missing genes are zero-filled
    (or raise if fill_missing='raise'). Cached as a torch tensor on first call.
    """
    if fill_missing not in ("zero", "raise"):
        raise ValueError(f"fill_missing must be 'zero' or 'raise', got {fill_missing!r}")

    cache_path = Path(cache_path)
    pickle_path = Path(pickle_path)

    if cache_path.exists():
        print(f"Loading cached GenePT map from {cache_path}")
        return torch.load(cache_path, weights_only=False)

    # Fetch + extract the pickle if we don't have it yet.
    if not pickle_path.exists():
        zip_path = pickle_path.with_name("GenePT_emebdding_v2.zip")
        if not zip_path.exists():
            print(f"Downloading GenePT v2 bundle (~574 MB) from {zenodo_url}")
            urllib.request.urlretrieve(zenodo_url, zip_path)
            print(f"  saved to {zip_path}")
        print(f"Extracting {pickle_basename} from {zip_path}")
        with zipfile.ZipFile(zip_path) as zf:
            # Some Zenodo bundles have a stray trailing dot in member names
            # (e.g. 'GenePT_gene_protein_embedding_model_3_text.pickle.') —
            # match by basename stripped of trailing dots.
            target = pickle_basename.rstrip(".")
            member = next(
                (m for m in zf.namelist() if Path(m).name.rstrip(".") == target),
                None,
            )
            if member is None:
                raise FileNotFoundError(
                    f"{pickle_basename} not in zip; members: {zf.namelist()[:10]}"
                )
            with zf.open(member) as src, open(pickle_path, "wb") as dst:
                dst.write(src.read())
        print(f"  extracted to {pickle_path}")

    print(f"Loading GenePT pickle from {pickle_path}")
    with open(pickle_path, "rb") as f:
        gene2vec = pickle.load(f)

    symbols = adata.var[symbol_key].astype(str).str.upper().tolist()
    d = int(np.asarray(next(iter(gene2vec.values()))).shape[0])  # 3072 for text-embedding-3-large
    sm = torch.zeros(len(symbols), d, dtype=torch.float32)
    missing = []
    for i, g in enumerate(symbols):
        v = gene2vec.get(g)
        if v is None:
            missing.append(g)
            continue
        sm[i] = torch.as_tensor(np.asarray(v, dtype=np.float32))

    if missing and fill_missing == "raise":
        raise KeyError(
            f"{len(missing)}/{len(symbols)} symbols missing from GenePT "
            f"(first 5: {missing[:5]})"
        )
    print(
        f"GenePT: {len(symbols) - len(missing)}/{len(symbols)} genes mapped, "
        f"{len(missing)} zero-filled (d={d})"
    )

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(sm, cache_path)
    print(f"  cached aligned map to {cache_path}")
    return sm


adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)
print("var cols:", list(adata.var.columns))

# Build/load the FULL-gene GenePT semantic map first, then subset along with adata.
semantic_map = get_or_build_genept_map(adata, SEMANTIC_CACHE, GENEPT_PICKLE_PATH)
print("semantic_map (raw):", tuple(semantic_map.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False
    )
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
    print(f"After HVG filter (top {HVG_TOP_N}, flavor={HVG_FLAVOR}):")
    print("  adata:", adata.shape)
    print("  semantic_map:", tuple(semantic_map.shape))
else:
    print("HVG filter skipped (HVG_TOP_N=None)")

In [ ]:
sc.pp.subsample(adata, n_obs=50000, random_state=42)

In [ ]:
# Each scvi model writes a UUID into adata.uns, so each needs its own copy.
adata_sem = adata.copy()
semantic_model = train_or_load_semantic_scvi(
    adata_sem,
    semantic_map,
    cache_dir=MODEL_CACHE_DIR / SEMANTIC_SCVI_SUBDIR,
    force_train=FORCE_TRAIN_SEMANTIC,
    max_epochs=SEMANTIC_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_N_EPOCHS_KL_WARMUP,
    **SEMANTIC_KWARGS,
)

adata_ldvae = adata.copy()
ldvae_model = train_or_load_nonneg_ldvae(
    adata_ldvae,
    cache_dir=MODEL_CACHE_DIR / "ldvae_nn",
    force_train=FORCE_TRAIN_LDVAE,
    max_epochs=LDVAE_MAX_EPOCHS,
    **LDVAE_KWARGS,
)

schpf_model = train_or_load_pickle(
    "scHPF",
    lambda: train_schpf_model(adata, n_factors=N_FACTORS),
    cache_path=MODEL_CACHE_DIR / "schpf_k10.pkl",
    force_train=FORCE_TRAIN_SCHPF,
)

def _train_nmf():
    print(f"Training cNMF (sklearn NMF, k={N_FACTORS}, init={NMF_INIT})...")
    X_nmf = adata.X
    if issparse(X_nmf):
        if (X_nmf.data < 0).any():
            X_nmf.data = np.maximum(X_nmf.data, 0)
    else:
        X_nmf = np.maximum(X_nmf, 0)
    _nmf = NMF(
        n_components=N_FACTORS,
        init=NMF_INIT,
        random_state=NMF_RANDOM_STATE,
        max_iter=NMF_MAX_ITER,
    )
    W = _nmf.fit_transform(X_nmf)
    H = _nmf.components_
    return NMFWrapper(model=_nmf, W=W, H=H, feature_names=adata.var_names)

nmf_model = train_or_load_pickle(
    "cNMF/NMF",
    _train_nmf,
    cache_path=MODEL_CACHE_DIR / "nmf_k10.pkl",
    force_train=FORCE_TRAIN_NMF,
)

In [ ]:
# ============================================================
# Training-loss curves — train/val for SemanticSCVI and LDVAE+.
# scHPF / cNMF have no per-epoch history, so they're skipped.
# ============================================================
import importlib, benchmark_helpers
importlib.reload(benchmark_helpers)
from benchmark_helpers import plot_training_curves

plot_training_curves(
    semantic_model, "SemanticSCVI",
    OUT_DIR / "training_curves_semantic.png",
    semantic=True,
)
plot_training_curves(
    ldvae_model, "LDVAE+",
    OUT_DIR / "training_curves_ldvae.png",
    semantic=False,
)
print("scHPF / cNMF: no per-epoch training history (skipping).")


In [ ]:
models = {
    MODEL_NAMES[0]: _ScviAdapter(semantic_model, adata_sem),
    MODEL_NAMES[1]: _ScviAdapter(ldvae_model, adata_ldvae),
    MODEL_NAMES[2]: schpf_model,
    MODEL_NAMES[3]: nmf_model,
}
for name, m in models.items():
    z = m.get_latent_representation()
    print(f"{name}: latent shape = {z.shape}")

In [ ]:
bench = SemanticBenchmark(
    models,
    adata,
    pathway_index=str(PATHWAY_INDEX),
    gene_mapping=GENE_MAPPING,
    out_dir=str(OUT_DIR),
)
bench.run_figures(
    semantic_map=semantic_map,
    lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
    lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
    per_projection_n_top=PER_PROJECTION_N_TOP,
    cluster_n_top=CLUSTER_N_TOP,
    cluster_top_per_cluster=CLUSTER_TOP_PER_CLUSTER,
)

In [ ]:
if ENABLE_LLM_GRADING:
    bench.run_grading(
        lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
        lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
        per_projection_n_top=PER_PROJECTION_N_TOP,
        cluster_n_top=CLUSTER_N_TOP,
    )
else:
    print("LLM grading skipped (ENABLE_LLM_GRADING=False)")

In [ ]:
from datetime import datetime

notes = (
    f"SemanticSCVI [GenePT-3-large prior]: {SEMANTIC_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_WARMUP_EPOCHS}/{SEMANTIC_MAX_EPOCHS} ep. "
    f"LDVAE+: weights_positive=True, {LDVAE_MAX_EPOCHS} ep. "
    f"scHPF: k={N_FACTORS}. cNMF: sklearn NMF k={N_FACTORS}, init={NMF_INIT}. "
    f"Benchmark: per_projection_n_top={PER_PROJECTION_N_TOP}, "
    f"cluster_n_top={CLUSTER_N_TOP}, cluster_top_per_cluster={CLUSTER_TOP_PER_CLUSTER}."
)
report_path = build_report(OUT_DIR, MODEL_NAMES, adata.shape, notes=notes)

# Rename to <dataset>_genept_<YYYYMMDD_HHMMSS>.html and drop intermediate artifacts.
# Preserved: LLM/score caches (`_llm_cache/`, `_score_cache/`) and prior .html reports.
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_name = f"{ADATA_PATH.stem}_genept_{ts}.html"
final_path = OUT_DIR / final_name
report_path.rename(final_path)

PRESERVE_DIRS = {"_llm_cache", "_score_cache"}
for child in OUT_DIR.iterdir():
    if child == final_path:
        continue
    if child.is_dir():
        if child.name in PRESERVE_DIRS:
            continue
        import shutil
        shutil.rmtree(child)
    else:
        if child.suffix == ".html":
            continue
        child.unlink()

print(f"Report written: {final_path}")
print(f"Size: {final_path.stat().st_size / 1e6:.1f} MB")